# CS781 Phase 6 — Notebook P6_02: Abstractive legal summarization with the Phase-1 models

Every Phase-1 model is trained to turn a **document chunk (≤ 600 tokens) into the summary of that chunk** (pairs
built by P6_01's alignment). A document's summary = the chunk summaries generated in order, trimmed to the length the
length model predicts. One notebook runs a queue of experiments (one per GPU); each writes its validation scores,
generated validation summaries and a ready `submission.zip`.

| Step | What happens |
|---|---|
| 1 | config: Phase-1 weights, experiments, queues |
| 2 | inputs: P6_01 pair files, tokenizer, Phase-1 checkpoints, earlier runs (KD teacher) |
| 3 | model classes (NB3) + adaptation methods (LoRA, LoRMA, adapters) |
| 4 | **sequence-to-sequence wrapper** for every architecture: decoder-only (A0/A1/F) and prefix-LM (E) read `chunk + " Summary:"` and continue with the summary; encoder-decoder (B) encodes the chunk and decodes the summary |
| 5 | **techniques**: LoRA, LoRMA⁺, Houlsby adapters (after attention and FFN), Pfeiffer adapters (after FFN only), full FT (reference), smaller students (`keep_layers`) and **word-level knowledge distillation** |
| 6 | **generation**: KV-cached greedy decoding, n-gram blocking, length control; chunk summaries joined in document order |
| 7 | training, validation (loss, ROUGE-1/L, redundancy, length), test generation → submission |
| 8 | unit tests | 9 | run the queue | 10 | results + **BERTScore** of every experiment's validation summaries (needs Internet) |

**Kaggle settings**: GPU **T4 x2**. Internet **ON** if you want BERTScore on validation (the training itself needs no Internet).
**Inputs**: (1) **P6_01 output**, (2) **NB1 output** (tokenizer), (3) the **Phase-1 weight datasets**, (4) for KD runs in a
later session: the run that trained the teacher.

In [1]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
  disk @ /tmp/cs781_scratch: free 1126.1 GB / total 8758.4 GB
HF token found: True


## 1. Configuration

In [2]:
import hashlib, torch, torch.nn as nn, torch.nn.functional as F, zipfile, threading, traceback
import pandas as pd
try: display
except NameError: display = print

P6_SMOKE = os.environ.get("CS781_P6_SMOKE") == "1"
P1_SOURCES = {
    "A0": dict(dataset="cs781-p1-model-a0-sc80", run_name="A0_gpt2_110M_SC80_epoch_1000M"),
    "A1": dict(dataset="cs781-p1-model-a1-sc80", run_name="A1_modern_110M_SC80_epoch_1000M"),
    "B":  dict(dataset="cs781-p1-model-b", run_name="B_encdec_6x6_SC80_epoch_1000M"),
    "E":  dict(dataset="data-partial-e-prefixlm-110m-sc80-epoch-1000m", run_name="E_prefixlm_110M_SC80_epoch_1000M"),
    "F":  dict(dataset="cs781-p1-model-f", run_name="F_localglobal_gqa_alibi_SC80_epoch_1000M"),
}
DEFAULTS = dict(
    model="E", init="pretrained", keep_layers=None, method="lorma", lr=None,
    lora_r=16, lora_alpha=32, lora_targets="all", lorma_variant="plus", adapter_dim=64,
    src_tok=600, tgt_tok=384, max_train_pairs=20000, drop_empty_targets=True,
    epochs=2, batch=4, accum=8, weight_decay=0.01, warmup=0.06, grad_clip=1.0, dropout=0.1,
    teacher=None, kd_alpha=0.5, kd_T=2.0,
    # generation
    gen_batch=24, no_repeat_ngram=4, rep_penalty=1.1, min_frac=0.3, max_frac=1.6, len_scale=1.1, n_val_gen=150,
    seed=0, est_h=None,
)
METHOD_LR = {"full": 5e-5, "lora": 5e-4, "lorma": 5e-4, "houlsby": 1e-3, "pfeiffer": 1e-3}
EXPERIMENTS = {
    # ---- account 1: the LoRMA model (also the KD teacher), LoRA, and the distilled student ----
    "E_lorma":          dict(model="E", method="lorma"),
    "A1_lora":          dict(model="A1", method="lora"),
    "A0_6L_kd":         dict(model="A0", keep_layers=6, method="full", lr=1e-4, teacher="E_lorma"),
    # ---- account 2: adapters on F and B, a full fine-tuning reference, the KD control ----
    "F_houlsby":        dict(model="F", method="houlsby"),
    "B_pfeiffer":       dict(model="B", method="pfeiffer"),
    "E_full":           dict(model="E", method="full"),
    "A0_6L_control":    dict(model="A0", keep_layers=6, method="full", lr=1e-4),
    # ---- decoding-only variants (no training; ~15 min each): how summary length / repetition control change the scores ----
    "E_lorma_len0.6":   dict(gen_from="E_lorma", len_scale=0.6, max_frac=1.0),
    "E_lorma_len1.6":   dict(gen_from="E_lorma", len_scale=1.6, min_frac=0.6, max_frac=2.2),
    "E_lorma_norep":    dict(gen_from="E_lorma", no_repeat_ngram=0, rep_penalty=1.0),
}
QUEUES = {"acc1": ["E_lorma", "A1_lora", "A0_6L_kd", "E_lorma_len0.6", "E_lorma_len1.6", "E_lorma_norep"],
          "acc2": ["F_houlsby", "E_full", "B_pfeiffer", "A0_6L_control"]}
RUN_QUEUE = QUEUES["acc2"]
BERTSCORE_EVAL = True          # final cell: BERTScore of all validation summaries (needs Internet; skipped if unavailable)

if P6_SMOKE:
    P1_SOURCES = {k: dict(dataset="", run_name=f"SMOKE_{k}") for k in P1_SOURCES}
    DEFAULTS.update(epochs=1, max_train_pairs=200, n_val_gen=8, gen_batch=4)
    for k in EXPERIMENTS:
        if EXPERIMENTS[k].get("keep_layers"): EXPERIMENTS[k]["keep_layers"] = 1
    RUN_QUEUE = [x for x in os.environ.get("CS781_P6_QUEUE", "E_lorma").split(",") if x]
    BERTSCORE_EVAL = False

def exp_cfg(name):
    cfg = dict(DEFAULTS); cfg.update(EXPERIMENTS[name]); cfg["name"] = name
    if cfg["lr"] is None: cfg["lr"] = METHOD_LR[cfg["method"]]
    if cfg["est_h"] is None: cfg["est_h"] = 0.3 if cfg.get("gen_from") else 0.6 * cfg["epochs"] * cfg["max_train_pairs"] / 20000 + 0.4
    return cfg

EXPS_DIR = ROOT / "p6" / "exps"; EXPS_DIR.mkdir(parents=True, exist_ok=True)
NUM_GPUS = torch.cuda.device_count()
print("GPUs:", NUM_GPUS, "| queue:", RUN_QUEUE)

GPUs: 2 | queue: ['F_houlsby', 'E_full', 'B_pfeiffer', 'A0_6L_control']


## 2. Inputs

In [3]:
META = load_json(find_artifact("p6/data/meta.json")) if find_artifact("p6/data/meta.json") else None
assert META is not None, "attach the P6_01 output (p6/data/…)"
from tokenizers import Tokenizer
TOK = Tokenizer.from_file(str(find_artifact("tokenizer/bpe32768/tokenizer.json") or "missing"))
EOS_ID, PAD_ID, BOS_ID = 0, 1, 2
SEP_IDS = TOK.encode(" Summary:").ids                          # separates the chunk from its summary (decoder-only / prefix-LM)
LEN_MODEL = META["len_model"]

class Pairs:
    def __init__(self, split):
        self.split = split
        def mm(kind):
            p = find_artifact(f"p6/data/{split}_{kind}_tokens.bin")
            if p is None: return None, None
            return np.memmap(p, dtype=np.uint16, mode="r"), np.load(find_artifact(f"p6/data/{split}_{kind}_offsets.npy"))
        self.src, self.src_off = mm("src"); self.tgt, self.tgt_off = mm("tgt")
        z = np.load(find_artifact(f"p6/data/{split}_pairs.npz")); self.doc, self.chunk, self.n_chunks = z["doc"], z["chunk"], z["n_chunks"]
        d = load_json(find_artifact(f"p6/data/{split}_docs.json")); self.ids, self.doc_words = d["ids"], np.array(d["doc_words"])
    def __len__(self): return len(self.doc)
    def s(self, i): return np.asarray(self.src[self.src_off[i]:self.src_off[i + 1]], dtype=np.int64)
    def t(self, i): return np.asarray(self.tgt[self.tgt_off[i]:self.tgt_off[i + 1]], dtype=np.int64)
    def src_len(self, i): return int(self.src_off[i + 1] - self.src_off[i])

PAIRS = {s: Pairs(s) for s in ("train", "val", "test")}
VAL_REFS = {r["id"]: r["summary"] for r in load_json(find_artifact("p6/data/val_refs.json"))}
for s, p in PAIRS.items():
    print(f"{s}: {len(p.ids)} documents, {len(p)} chunks" + (f", empty targets {np.mean(np.diff(p.tgt_off) == 0):.1%}" if p.tgt is not None else ""))

INPUT_ROOTS = [os.environ["CS781_P6_INPUTS"]] if os.environ.get("CS781_P6_INPUTS") else (["/kaggle/input"] if IN_KAGGLE else [])
EXP_DIRS = {}
for r in INPUT_ROOTS:
    for f in sorted(glob.glob(f"{r}/**/result.json", recursive=True) + glob.glob(f"{r}/**/trainable.pt", recursive=True)):
        p = Path(f).parent
        if p.name in EXPERIMENTS: EXP_DIRS.setdefault(p.name, []).append(p)
def find_exp_file(name, fname):
    p = EXPS_DIR / name / fname
    if p.exists(): return p
    for d in EXP_DIRS.get(name, []):
        if (d / fname).exists(): return d / fname
    return None
for name, dirs in EXP_DIRS.items():                              # restore small result files of attached earlier runs
    for dd in dirs:
        for src in dd.glob("*"):
            if src.is_file() and src.suffix in (".json", ".zip", ".txt") and src.name != "error.txt":
                dst = EXPS_DIR / name / src.name
                if not dst.exists(): dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
if EXP_DIRS: print("attached earlier experiments:", sorted(EXP_DIRS))

def find_p1_dir(key):
    src = P1_SOURCES[key]
    roots = [os.environ["CS781_P1_RUNS"]] if os.environ.get("CS781_P1_RUNS") else []
    roots += [str(ROOT / "runs")] + (["/kaggle/input"] if IN_KAGGLE else [])
    cands = sorted({c for r in roots for c in glob.glob(f"{r}/**/weights_final.pt", recursive=True)})
    by_run = [c for c in cands if Path(c).parent.name == src["run_name"]]
    by_ds = [c for c in cands if src["dataset"] and src["dataset"].lower() in [p.lower() for p in Path(c).parts]]
    hit = (by_run or by_ds or [None])[0]
    return Path(hit).parent if hit else None
P1_DIRS = {k: find_p1_dir(k) for k in P1_SOURCES}
for k, p in P1_DIRS.items(): print(f"  {k}: {p if p else 'NOT FOUND'}")

train: 6646 documents, 62491 chunks, empty targets 23.9%
val: 355 documents, 3300 chunks, empty targets 23.3%
test: 100 documents, 1020 chunks
  A0: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a0-sc80/cs781_p1/runs/A0_gpt2_110M_SC80_epoch_1000M
  A1: /kaggle/input/datasets/aarinsheth/cs781-p1-model-a1-sc80/cs781_p1/runs/A1_modern_110M_SC80_epoch_1000M
  B: /kaggle/input/datasets/aaravoswal/cs781-p1-model-b/cs781_p1/runs/B_encdec_6x6_SC80_epoch_1000M
  E: /kaggle/input/datasets/chrishitha2403/data-partial-e-prefixlm-110m-sc80-epoch-1000m/cs781_p1/runs/E_prefixlm_110M_SC80_epoch_1000M
  F: /kaggle/input/datasets/chrishitha2403/cs781-p1-model-f/cs781_p1/runs/F_localglobal_gqa_alibi_SC80_epoch_1000M


## 3. Model classes (NB3) and adaptation methods (P3_02)

In [4]:
# (model classes copied unchanged from NB3's model cell)
class CausalSelfAttention(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.h, self.hd = h, d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.proj.RESIDUAL_SCALE = True
        self.dropout = dropout

    def forward(self, x, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, self.hd).transpose(1, 2) for t in (q, k, v)]
        if past is not None:
            k = torch.cat([past[0], k], dim=2); v = torch.cat([past[1], v], dim=2)
        present = (k, v) if use_cache else None
        drop = self.dropout if self.training else 0.0
        if mask is None:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=drop)
        else:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop)
        return self.proj(y.transpose(1, 2).reshape(B, T, C)), present

class MLP(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.proj.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class Block(nn.Module):
    def __init__(self, d, h, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn, self.mlp = CausalSelfAttention(d, h, dropout), MLP(d, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.ln1(x), past, mask, use_cache)
        x = x + self.drop(a)
        x = x + self.mlp(self.ln2(x))
        return x, present

class GPT2LM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte, self.wpe = nn.Embedding(vocab, d_model), nn.Embedding(ctx_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight                     # weight tying
        for name, m in self.named_modules():
            if isinstance(m, nn.Linear):
                std = 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02
                nn.init.normal_(m.weight, 0.0, std)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.drop(self.wte(idx) + self.wpe(pos))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.ln_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ A1: modern decoder
class RMSNorm(nn.Module):
    """x / sqrt(mean(x²)) · g — no mean subtraction, no bias (cheaper than LayerNorm, equally stable). Computed in fp32."""
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight).to(x.dtype)

def rope_tables(head_dim, n_pos, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    ang = torch.outer(torch.arange(n_pos).float(), inv)                    # (n_pos, head_dim/2)
    return torch.cos(ang), torch.sin(ang)

def apply_rope(x, cos, sin):
    """Rotate each (x1_i, x2_i) pair of the head dimension by angle pos·θ_i. The dot product q·k then depends only on
    the RELATIVE distance between the two positions. x: (B, H, T, D); cos/sin: (B or 1, 1, T, D/2)."""
    d = x.size(-1) // 2
    x1, x2 = x[..., :d].float(), x[..., d:].float()
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).to(x.dtype)

class ModernAttention(nn.Module):
    def __init__(self, d, h, kv, dropout, rope, causal=True):
        super().__init__()
        self.h, self.kv, self.hd, self.rope, self.dropout, self.causal = h, kv, d // h, rope, dropout, causal
        self.q = nn.Linear(d, d, bias=False)
        self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)             # GQA when kv < h: fewer key/value heads
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if self.rope:
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        drop = self.dropout if self.training else 0.0
        # no explicit mask: causal for decoders (but a single cached query may see every past key), full for encoders
        y = (F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and past is None, dropout_p=drop)
             if mask is None else F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=drop))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class SwiGLU(nn.Module):
    """out = W2( SiLU(W1 x) ⊙ W3 x ): a learned gate decides, per hidden unit, how much of W3 x passes through."""
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False); self.w2.RESIDUAL_SCALE = True
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class GeluMLP(nn.Module):
    def __init__(self, d, hidden, dropout):
        super().__init__()
        self.fc, self.proj = nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False)
        self.proj.RESIDUAL_SCALE = True; self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x), approximate="tanh")))

class ModernBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, norm, mlp, rope):
        super().__init__()
        Norm = RMSNorm if norm == "rms" else nn.LayerNorm
        self.n1, self.n2 = Norm(d), Norm(d)
        self.attn = ModernAttention(d, h, kv, dropout, rope)
        self.mlp = SwiGLU(d, hidden, dropout) if mlp == "swiglu" else GeluMLP(d, hidden, dropout)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, cos, sin, past=None, mask=None, use_cache=False):
        a, present = self.attn(self.n1(x), cos, sin, past, mask, use_cache)
        x = x + self.drop(a)
        return x + self.mlp(self.n2(x)), present

class ModernLM(nn.Module):
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=None, ffn_hidden=2048,
                 rope_base=10000, pos="rope", norm="rms", mlp="swiglu", **_):
        super().__init__()
        self.ctx_len, self.use_rope = ctx_len, pos == "rope"
        self.wte = nn.Embedding(vocab, d_model)
        self.wpe = None if self.use_rope else nn.Embedding(ctx_len, d_model)
        hd = d_model // n_head
        cos, sin = rope_tables(hd, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, norm, mlp,
                                                 self.use_rope) for _ in range(n_layer)])
        self.norm_f = RMSNorm(d_model) if norm == "rms" else nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False)
        self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        x = self.wte(idx)
        if self.wpe is not None: x = x + self.wpe(pos)
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        x = self.drop(x)
        new_cache = []
        for i, blk in enumerate(self.blocks):
            x, present = blk(x, cos, sin, cache[i] if cache is not None else None, mask, use_cache)
            new_cache.append(present)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new_cache if use_cache else None)

# ------------------------------------------------------------------ B: encoder-decoder
class CrossAttention(nn.Module):
    """decoder queries attend to the encoder's output ("memory"); keys/values are computed once per prompt"""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, mem, mem_valid=None, kv=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        if kv is None:
            S = mem.size(1)
            kv = tuple(self.kvp(mem).view(B, S, 2, self.h, self.hd).permute(2, 0, 3, 1, 4))
        k, v = kv
        mask = None if mem_valid is None else mem_valid[:, None, None, :]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o(y.transpose(1, 2).reshape(B, T, C)), kv

class EncoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn = ModernAttention(d, h, h, dropout, rope=True, causal=False)     # bidirectional
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mask=None):
        x = x + self.attn(self.n1(x), cos, sin, None, mask, False)[0]
        return x + self.mlp(self.n2(x))

class DecoderBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2, self.n3 = RMSNorm(d), RMSNorm(d), RMSNorm(d)
        self.self_attn = ModernAttention(d, h, h, dropout, rope=True, causal=True)
        self.cross = CrossAttention(d, h)
        self.mlp = SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, mem, mem_valid, cache=None, use_cache=False):
        a, self_kv = self.self_attn(self.n1(x), cos, sin, cache["self"] if cache else None, None, use_cache)
        x = x + a
        c, cross_kv = self.cross(self.n2(x), mem, mem_valid, cache["cross"] if cache else None)
        x = x + c
        return x + self.mlp(self.n3(x)), ({"self": self_kv, "cross": cross_kv} if use_cache else None)

class EncDecRunner:
    """generation for the encoder-decoder: encode the prompt once, then decode from <|bos|> one token at a time"""
    def __init__(self, model, rows, pad_id, device, amp):
        self.model, self.device, self.amp = model, device, amp
        B, L = len(rows), max(len(r) for r in rows)
        idx = torch.full((B, L), pad_id, dtype=torch.long); valid = torch.zeros((B, L), dtype=torch.bool)
        for r, ids_ in enumerate(rows):                                        # right padding (bidirectional encoder)
            idx[r, :len(ids_)] = torch.tensor(ids_); valid[r, :len(ids_)] = True
        self.idx, self.mem_valid = idx.to(device), valid.to(device)
    def prefill(self):
        with self.amp:
            self.mem = self.model.encode(self.idx, self.mem_valid)
            bos = torch.full((self.idx.size(0), 1), self.model.bos_id, dtype=torch.long, device=self.device)
            logits, self.cache = self.model.decode(bos, self.mem, self.mem_valid, 0, None, use_cache=True)
        self.t = 0
        return logits[:, -1].float()
    def step(self, nxt):
        self.t += 1
        with self.amp:
            logits, self.cache = self.model.decode(nxt[:, None], self.mem, self.mem_valid, self.t, self.cache, use_cache=True)
        return logits[:, -1].float()
    def reorder(self, idx):
        self.mem, self.mem_valid = self.mem[idx], self.mem_valid[idx]; self.cache = reorder_cache(self.cache, idx)

class EncDecLM(nn.Module):
    """B: Transformer encoder-decoder. Training: each window is split at a random point E — the encoder reads tokens
    [0, E) bidirectionally, the decoder (starting from <|bos|>, cross-attending to the encoder) predicts tokens [E, T].
    At test time the prompt goes to the encoder and the continuation is decoded — exactly the task's shape."""
    is_encdec = True
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=2048, rope_base=10000,
                 enc_layers=None, enc_len_eval=768, bos_id=2, **_):
        super().__init__()
        ne = enc_layers or n_layer // 2; nd = n_layer - ne
        self.bos_id, self.enc_len_eval, self.ctx_len = bos_id, enc_len_eval, ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.enc = nn.ModuleList([EncoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(ne)])
        self.dec = nn.ModuleList([DecoderBlock(d_model, n_head, ffn_hidden, dropout) for _ in range(nd)])
        self.enc_norm, self.norm_f = RMSNorm(d_model), RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _rope(self, start, T, device):
        p = torch.arange(start, start + T, device=device)[None]
        return self.rope_cos[p][:, None], self.rope_sin[p][:, None]
    def encode(self, idx, valid=None):
        cos, sin = self._rope(0, idx.size(1), idx.device)
        x = self.wte(idx); mask = None if valid is None else valid[:, None, None, :]
        for blk in self.enc: x = blk(x, cos, sin, mask)
        return self.enc_norm(x)
    def decode(self, idx, mem, mem_valid, start, cache=None, use_cache=False):
        cos, sin = self._rope(start, idx.size(1), idx.device)
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.dec):
            x, c = blk(x, cos, sin, mem, mem_valid, cache[i] if cache else None, use_cache); new.append(c)
        return self.lm_head(self.norm_f(x)), (new if use_cache else None)
    def forward(self, idx, targets=None, enc_len=None, **_):
        B, T = idx.shape
        E = int(enc_len or min(self.enc_len_eval, T - 1))
        mem = self.encode(idx[:, :E])
        dec_in = torch.cat([torch.full((B, 1), self.bos_id, dtype=torch.long, device=idx.device), idx[:, E:]], 1)
        logits, _ = self.decode(dec_in, mem, None, 0)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return EncDecRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ C: retention network (RetNet-style), D: hybrid
class Retention(nn.Module):
    """Multi-scale retention (Sun et al. 2023, "Retentive Network"). Attention without softmax:
         parallel form (training):   O = (Q Kᵀ ⊙ D) V,   D[n,m] = γ^(n−m) for n ≥ m, else 0
         recurrent form (generation): S_n = γ·S_{n−1} + k_nᵀ v_n,   o_n = q_n S_n
       Both give identical outputs. Each head has its own decay γ (short- to long-range memory), so generation needs
       only a fixed-size state per head — no growing KV-cache, O(1) cost per new token.
       Per-head normalisation + a SiLU output gate as in RetNet; RoPE on q/k for relative position."""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.hd = h, d // h
        self.q, self.k = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.v, self.g = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
        self.register_buffer("log_gamma", torch.log(1 - 2.0 ** (-5 - torch.arange(h, dtype=torch.float32))), persistent=False)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        B, T, C = x.shape
        q, k, v = [m(x).view(B, T, self.h, self.hd).transpose(1, 2) for m in (self.q, self.k, self.v)]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if valid is not None:                                   # left padding: padded positions contribute nothing
            vm = valid[:, None, :, None].to(k.dtype); k, v = k * vm, v * vm
        q = q * self.hd ** -0.5
        lg = self.log_gamma.view(1, self.h, 1, 1)
        new_state = None
        if state is None:                                       # parallel form
            n = torch.arange(T, device=x.device, dtype=torch.float32)
            diff = n[:, None] - n[None, :]
            D = torch.where(diff >= 0, torch.exp(diff * lg), torch.zeros((), device=x.device)).to(q.dtype)
            o = ((q @ k.transpose(-1, -2)) * D) @ v
            if use_cache:
                w = torch.exp((T - 1 - n).view(1, 1, T, 1) * lg)                       # γ^(T−1−m)
                new_state = (k.float() * w).transpose(-1, -2) @ v.float()            # (B, H, hd, hd)
        else:                                                   # recurrent form, one token at a time
            new_state = state * torch.exp(lg) + k.float().transpose(-1, -2) @ v.float()
            o = (q.float() @ new_state).to(q.dtype)
        o = F.layer_norm(o.float(), (self.hd,)).to(x.dtype)     # per-head normalisation (GroupNorm, no affine)
        o = o.transpose(1, 2).reshape(B, T, C)
        return self.o(F.silu(self.g(x)) * o), (new_state if use_cache else None)

class RetBlock(nn.Module):
    def __init__(self, d, h, hidden, dropout):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.ret, self.mlp = Retention(d, h), SwiGLU(d, hidden, dropout)
    def forward(self, x, cos, sin, state=None, valid=None, use_cache=False):
        r, s = self.ret(self.n1(x), cos, sin, state, valid, use_cache)
        x = x + r
        return x + self.mlp(self.n2(x)), s

class RetNetLM(nn.Module):
    """C (attn_every=0): every block is retention.  D (attn_every=k): every k-th block is full softmax attention
    (ModernBlock), the rest retention — retention supplies cheap long-range decayed memory, the few attention layers
    give exact recall of specific earlier tokens (names, section numbers)."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, ffn_hidden=1792, rope_base=10000,
                 attn_every=0, n_kv_head=None, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        cos, sin = rope_tables(d_model // n_head, 2 * ctx_len, rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
        self.kinds = ["attn" if attn_every and (i + 1) % attn_every == 0 else "ret" for i in range(n_layer)]
        self.blocks = nn.ModuleList([
            ModernBlock(d_model, n_head, n_kv_head or n_head, ffn_hidden, dropout, "rms", "swiglu", True) if kd == "attn"
            else RetBlock(d_model, n_head, ffn_hidden, dropout) for kd in self.kinds])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if pos is None: pos = torch.arange(T, device=idx.device)[None]
        cos, sin = self.rope_cos[pos][:, None], self.rope_sin[pos][:, None]
        valid = mask[:, 0, -1, :] if (mask is not None and cache is None) else None   # key validity at prefill
        x = self.wte(idx); new = []
        for i, (kd, blk) in enumerate(zip(self.kinds, self.blocks)):
            c = cache[i] if cache is not None else None
            if kd == "attn": x, c = blk(x, cos, sin, c, mask, use_cache)
            else:            x, c = blk(x, cos, sin, c, valid, use_cache)
            new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

# ------------------------------------------------------------------ E: prefix-LM (encoder/hybrid adapted for LM)
class PrefixLM(ModernLM):
    """E: the A1 stack with a different ATTENTION MASK. Prompt (prefix) tokens attend to the whole prompt in both
    directions, like a BERT encoder; continuation tokens attend causally, like GPT. One shared stack plays both roles.
    Training: split each window at a random E, loss only on the continuation (predictions from position E−1 on).
    Generation: prefill the prompt with the bidirectional mask, then ordinary cached causal decoding."""
    def __init__(self, *a, enc_len_eval=768, **kw):
        super().__init__(*a, **kw); self.enc_len_eval = enc_len_eval
    def forward(self, idx, targets=None, enc_len=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        if targets is None:                                       # generation path (mask supplied by the runner)
            return super().forward(idx, None, pos, mask, cache, use_cache, last_only)
        T = idx.size(1); E = int(enc_len or min(self.enc_len_eval, T - 1))
        i = torch.arange(T, device=idx.device)
        allowed = (i[None, :] <= i[:, None]) | ((i[:, None] < E) & (i[None, :] < E))   # causal ∪ (prefix × prefix)
        logits, _, _ = super().forward(idx, None, None, allowed[None, None], None, False, False)
        loss = F.cross_entropy(logits[:, E - 1:].float().reshape(-1, logits.size(-1)), targets[:, E - 1:].reshape(-1))
        return logits, loss, None
    def make_runner(self, rows, pad_id, device, amp):
        return PrefixRunner(self, rows, pad_id, device, amp)

# ------------------------------------------------------------------ F: efficient local-global decoder
def alibi_slopes(n):
    """ALiBi head slopes (Press et al. 2022): geometric sequence; extra heads interleaved when n is not a power of 2"""
    def pow2(m): return [2 ** (-8 * (k + 1) / m) for k in range(m)]
    if math.log2(n).is_integer(): return pow2(n)
    p = 2 ** math.floor(math.log2(n))
    return pow2(p) + pow2(2 * p)[0::2][: n - p]

class LocalGlobalAttention(nn.Module):
    """Grouped-query attention (h query heads share kv key/value heads) with an ALiBi distance penalty added to the
    attention logits (no position embeddings at all). window=W makes it sliding-window attention: a query only sees
    the previous W tokens; window=None is full causal attention."""
    def __init__(self, d, h, kv, window):
        super().__init__()
        self.h, self.kv, self.hd, self.window = h, kv, d // h, window
        self.q = nn.Linear(d, d, bias=False); self.kvp = nn.Linear(d, 2 * kv * self.hd, bias=False)
        self.o = nn.Linear(d, d, bias=False); self.o.RESIDUAL_SCALE = True
    def forward(self, x, bias, past=None, use_cache=False):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k, v = self.kvp(x).view(B, T, 2, self.kv, self.hd).permute(2, 0, 3, 1, 4)
        if past is not None:
            k = torch.cat([past[0], k], 2); v = torch.cat([past[1], v], 2)
        present = (k, v) if use_cache else None
        if self.kv != self.h:
            k, v = k.repeat_interleave(self.h // self.kv, 1), v.repeat_interleave(self.h // self.kv, 1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype))
        return self.o(y.transpose(1, 2).reshape(B, T, C)), present

class EffBlock(nn.Module):
    def __init__(self, d, h, kv, hidden, dropout, window):
        super().__init__()
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.attn, self.mlp = LocalGlobalAttention(d, h, kv, window), SwiGLU(d, hidden, dropout)
    def forward(self, x, bias, past=None, use_cache=False):
        a, present = self.attn(self.n1(x), bias, past, use_cache)
        x = x + a
        return x + self.mlp(self.n2(x)), present

class EfficientLM(nn.Module):
    """F: lightweight decoder. Sliding-window attention (window W) in most layers, full causal attention in every
    k-th layer, grouped-query attention (fewer key/value heads → smaller KV-cache), ALiBi instead of position
    embeddings. Local layers model nearby text; the few global layers can still reach any earlier token."""
    def __init__(self, vocab, n_layer, n_head, d_model, ctx_len, dropout=0.0, n_kv_head=4, ffn_hidden=2304,
                 window=256, global_every=4, **_):
        super().__init__()
        self.ctx_len = ctx_len
        self.wte = nn.Embedding(vocab, d_model)
        self.windows = [None if global_every and (i + 1) % global_every == 0 else window for i in range(n_layer)]
        self.blocks = nn.ModuleList([EffBlock(d_model, n_head, n_kv_head, ffn_hidden, dropout, w) for w in self.windows])
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head)), persistent=False)
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab, bias=False); self.lm_head.weight = self.wte.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * n_layer) if getattr(m, "RESIDUAL_SCALE", False) else 0.02)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0.0, 0.02)
    def _bias(self, qpos, kpos, key_ok, window):
        """(B|1, H, Tq, Tk) additive mask: −slope·distance for allowed pairs, −inf elsewhere"""
        dist = (qpos[:, :, None] - kpos[:, None, :]).float()               # (B|1, Tq, Tk)
        ok = (dist >= 0) & key_ok[:, None, :]
        if window is not None: ok = ok & (dist < window)
        b = -self.slopes.view(1, -1, 1, 1) * dist[:, None]
        return b.masked_fill(~ok[:, None], float("-inf"))
    def forward(self, idx, targets=None, pos=None, mask=None, cache=None, use_cache=False, last_only=False):
        B, T = idx.shape
        if mask is None:                                          # training / plain forward: positions 0..T-1
            qpos = kpos = torch.arange(T, device=idx.device)[None]
            key_ok = torch.ones((1, T), dtype=torch.bool, device=idx.device)
        else:                                                     # generation: derive key positions from the mask
            key_ok = mask[:, 0, -1, :]                            # validity of every key (past + current)
            kpos = key_ok.long().cumsum(-1) - 1
            qpos = pos
        biases = {w: self._bias(qpos, kpos, key_ok, w) for w in set(self.windows)}
        x = self.wte(idx); new = []
        for i, blk in enumerate(self.blocks):
            x, c = blk(x, biases[self.windows[i]], cache[i] if cache is not None else None, use_cache); new.append(c)
        x = self.norm_f(x)
        if last_only: x = x[:, -1:]
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss, (new if use_cache else None)

MODEL_REGISTRY = {"gpt2": GPT2LM, "modern": ModernLM, "encdec": EncDecLM, "retnet": RetNetLM, "hybrid": RetNetLM,
                  "prefixlm": PrefixLM, "efficient": EfficientLM}

def build_model(cfg, vocab):
    return MODEL_REGISTRY[cfg["arch"]](vocab=vocab, **cfg)

def count_params(m):
    return sum(p.numel() for p in m.parameters())      # tied weights counted once

In [5]:
# (backbone + adaptation methods copied unchanged from P3_02)
class Backbone(nn.Module):
    def __init__(self, lm, arch, attn):
        super().__init__()
        self.lm, self.arch = lm, arch
        self.bidir = attn == "bidir" or arch == "encdec"
        self.d = lm.wte.weight.shape[1]
        if arch == "encdec": lm.dec = nn.ModuleList()
        self.ctx_limit = lm.ctx_len if arch == "gpt2" else (2 * lm.ctx_len if arch in ("modern", "prefixlm", "encdec") else 10 ** 9)
    def blocks(self):
        return self.lm.enc if self.arch == "encdec" else self.lm.blocks
    def forward(self, idx, valid, prefix=None):
        lm, (B, T), dev = self.lm, idx.shape, idx.device
        P = 0 if prefix is None else prefix.size(0)
        x = lm.wte(idx)
        if P:
            x = torch.cat([prefix[None].expand(B, -1, -1).to(x.dtype), x], 1)
            valid = torch.cat([torch.ones(B, P, dtype=torch.bool, device=dev), valid], 1)
        L = T + P
        assert L <= self.ctx_limit, f"window {L} longer than the model's position range {self.ctx_limit}"
        pos = torch.arange(L, device=dev)[None]
        mask = valid[:, None, None, :] if self.bidir else None
        if self.arch == "encdec":
            cos, sin = lm._rope(0, L, dev)
            for blk in lm.enc: x = blk(x, cos, sin, mask)
            h = lm.enc_norm(x)
        elif self.arch == "gpt2":
            x = lm.drop(x + lm.wpe(pos))
            for blk in lm.blocks: x, _ = blk(x, None, mask, False)
            h = lm.ln_f(x)
        elif self.arch in ("modern", "prefixlm"):
            if lm.wpe is not None: x = x + lm.wpe(pos)
            cos, sin = lm.rope_cos[pos][:, None], lm.rope_sin[pos][:, None]
            x = lm.drop(x)
            for blk in lm.blocks: x, _ = blk(x, cos, sin, None, mask, False)
            h = lm.norm_f(x)
        elif self.arch == "efficient":
            dist = (pos[:, :, None] - pos[:, None, :]).float()
            biases = {}
            for w in set(lm.windows):
                if self.bidir:
                    ok = valid[:, None, :] & ((dist.abs() < w) if w is not None else torch.ones_like(dist, dtype=torch.bool))
                    ok = ok | torch.eye(L, dtype=torch.bool, device=dev)[None]
                    b = -lm.slopes.view(1, -1, 1, 1) * dist.abs()[:, None]
                else:
                    ok = dist >= 0
                    if w is not None: ok = ok & (dist < w)
                    b = -lm.slopes.view(1, -1, 1, 1) * dist[:, None]
                biases[w] = b.masked_fill(~ok[:, None], float("-inf"))
            for i, blk in enumerate(lm.blocks): x, _ = blk(x, biases[lm.windows[i]], None, False)
            h = lm.norm_f(x)
        else:
            raise ValueError(self.arch)
        return h[:, P:]

class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.scale = base, alpha / (math.sqrt(r) if rs else r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

_PERM_IDX, _PERM_LOCK = {}, threading.Lock()
def perm_index(d, device):
    key = (d, str(device))
    with _PERM_LOCK:
        if key not in _PERM_IDX:
            i = torch.arange(d, device=device); _PERM_IDX[key] = (i[None, :] - i[:, None]) % d
        return _PERM_IDX[key]

class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha, variant="plus"):
        super().__init__()
        d = base.out_features; self.base, self.variant = base, variant
        if variant == "plus":
            self.scale = alpha / r
            self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B = nn.Parameter(torch.zeros(d, r))
        else:
            A = torch.zeros(r, d); A[0, 0] = 1.0
            B = torch.empty(d, r); nn.init.kaiming_uniform_(B, a=math.sqrt(5)); B[:, 0] = 1.0
            self.A, self.B = nn.Parameter(A), nn.Parameter(B)
    def forward(self, x):
        y = self.base(x)
        if self.variant == "plus":
            return y + self.scale * F.linear(F.linear(y, self.A), self.B)
        M = self.B @ self.A
        return F.linear(y, M.gather(1, perm_index(M.size(0), M.device)))

class IA3Linear(nn.Module):
    """(IA)^3: mode "out" rescales the layer's outputs (keys/values; `mask` limits it to part of a fused q|k|v
    output), mode "in" rescales its inputs (= the FFN hidden activation when wrapped around the down-projection)"""
    def __init__(self, base, mode, mask=None):
        super().__init__()
        self.base, self.mode = base, mode
        n = base.out_features if mode == "out" else base.in_features
        self.l = nn.Parameter(torch.ones(n))
        self.register_buffer("mask", mask if mask is not None else torch.ones(n, dtype=torch.bool), persistent=False)
    def forward(self, x):
        if self.mode == "in": return self.base(x * self.l.to(x.dtype))
        s = torch.where(self.mask, self.l, torch.ones_like(self.l))
        return self.base(x) * s.to(x.dtype)

class BiasLinear(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.bias = nn.Parameter(torch.zeros(base.out_features))
    def forward(self, x): return self.base(x) + self.bias

class AdapterLinear(nn.Module):
    def __init__(self, base, dim):
        super().__init__()
        d = base.out_features; self.base = base
        self.down, self.up = nn.Linear(d, dim), nn.Linear(dim, d)
        nn.init.normal_(self.down.weight, 0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.base(x); return y + self.up(F.gelu(self.down(y)))

ATTN_TYPES = (CausalSelfAttention, ModernAttention, LocalGlobalAttention)
MLP_TYPES = (MLP, SwiGLU, GeluMLP)

def target_linears(backbone, which):
    out = []
    for parent in list(backbone.blocks().modules()):
        kind = "attn" if isinstance(parent, ATTN_TYPES) else "mlp" if isinstance(parent, MLP_TYPES) else None
        if kind is None or (which == "attn" and kind != "attn"): continue
        for name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear):
                out.append((parent, name, child, bool(getattr(child, "RESIDUAL_SCALE", False))))
    return out

def apply_method(backbone, cfg):
    m = cfg["method"]
    for p in backbone.parameters(): p.requires_grad = (m == "full")
    if m in ("full", "frozen", "prompt"): return
    if m == "topk":
        for blk in list(backbone.blocks())[-cfg["topk"]:]:
            for p in blk.parameters(): p.requires_grad = True
        return
    if m == "ia3":
        for parent in list(backbone.blocks().modules()):
            if isinstance(parent, CausalSelfAttention):                      # fused q|k|v: rescale only k and v
                d = parent.qkv.in_features
                parent.qkv = IA3Linear(parent.qkv, "out", torch.arange(3 * d) >= d)
            elif isinstance(parent, (ModernAttention, LocalGlobalAttention)):
                parent.kvp = IA3Linear(parent.kvp, "out")
            elif isinstance(parent, SwiGLU): parent.w2 = IA3Linear(parent.w2, "in")
            elif isinstance(parent, (MLP, GeluMLP)): parent.proj = IA3Linear(parent.proj, "in")
        return
    which = cfg["lora_targets"] if m in ("lora", "rslora", "lorma") else "all"
    for parent, name, lin, is_out in target_linears(backbone, which):
        if m in ("lora", "rslora"): setattr(parent, name, LoRALinear(lin, cfg["lora_r"], cfg["lora_alpha"], rs=(m == "rslora")))
        elif m == "lorma": setattr(parent, name, LoRMALinear(lin, cfg["lora_r"], cfg["lora_alpha"], cfg["lorma_variant"]))
        elif m == "adapter":
            if is_out: setattr(parent, name, AdapterLinear(lin, cfg["adapter_dim"]))
        elif m == "bitfit":
            if lin.bias is not None: lin.bias.requires_grad = True
            else: setattr(parent, name, BiasLinear(lin))
    if m == "bitfit":
        for mod in backbone.modules():
            if isinstance(mod, (nn.LayerNorm, RMSNorm)):
                for p in mod.parameters(): p.requires_grad = True

def set_backbone_dropout(backbone, p):
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p
        if isinstance(mod, (CausalSelfAttention, ModernAttention)): mod.dropout = p

_P1_CACHE, _P1_LOCK = {}, threading.Lock()
def load_phase1(key):
    with _P1_LOCK:
        if key not in _P1_CACHE:
            d = P1_DIRS[key]
            info = load_json(d / "run_info.json") or load_json(d / "summary.json") or {}
            pcfg = dict(info.get("cfg") or {}); assert pcfg, f"no run_info.json/summary.json next to {d}"
            sd = torch.load(d / "weights_final.pt", map_location="cpu", weights_only=False)
            _P1_CACHE[key] = (pcfg, {k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
        return _P1_CACHE[key]

## 4. Sequence-to-sequence wrapper
| architecture | training sequence | loss on |
|---|---|---|
| A0 / A1 / F (decoder) | `chunk  Summary:  summary <eos>` (causal) | summary tokens + `<eos>` |
| E (prefix-LM) | same, but `chunk Summary:` is a **bidirectional prefix** (as in E's pretraining) | summary tokens + `<eos>` |
| B (encoder-decoder) | encoder: chunk; decoder: `<bos> summary` | summary tokens + `<eos>` |

**Adapters** (`AdapterLinear`: down-projection → GELU → up-projection, zero-initialised, added to a layer's output):
Houlsby et al. 2019 place one after the attention output and one after the FFN output of every block; Pfeiffer et
al. 2020 keep only the FFN one (half the parameters). Our blocks are pre-norm; the adapter wraps the output
projection, i.e. it acts on the sub-layer output before the residual addition.

In [6]:
class S2S(nn.Module):
    def __init__(self, lm, arch):
        super().__init__(); self.lm, self.arch = lm, arch
        self.enc_dec = arch == "encdec"
    def blocks(self):
        return nn.ModuleList(list(self.lm.enc) + list(self.lm.dec)) if self.enc_dec else self.lm.blocks
    def logits(self, b):
        """teacher-forced logits (B, T, V) aligned with b['labels']"""
        lm = self.lm
        if self.enc_dec:
            mem = lm.encode(b["src"], b["src_valid"])
            return lm.decode(b["dec_in"], mem, b["src_valid"], 0)[0]
        if self.arch == "prefixlm":
            return ModernLM.forward(lm, b["idx"], None, None, b["mask"], None, False, False)[0]
        return lm(b["idx"])[0]

def make_batch(pairs, items, cfg, arch, device):
    """items: list of (src_ids, tgt_ids) with tgt possibly empty; returns tensors for S2S.logits + labels (-100 = no loss)"""
    srcs = [s[:cfg["src_tok"]] for s, _ in items]; tgts = [t[:cfg["tgt_tok"]] for _, t in items]
    T_ = lambda a: torch.from_numpy(a).to(device, non_blocking=True)
    if arch == "encdec":
        Ls, Lt = max(len(s) for s in srcs), max(len(t) for t in tgts) + 1
        src = np.full((len(items), Ls), PAD_ID, np.int64); sv = np.zeros((len(items), Ls), bool)
        dec = np.full((len(items), Lt), PAD_ID, np.int64); lab = np.full((len(items), Lt), -100, np.int64)
        for k, (s, t) in enumerate(zip(srcs, tgts)):
            src[k, :len(s)] = s; sv[k, :len(s)] = True
            dec[k, 0] = BOS_ID; dec[k, 1:len(t) + 1] = t; lab[k, :len(t)] = t; lab[k, len(t)] = EOS_ID
        return {"src": T_(src), "src_valid": T_(sv), "dec_in": T_(dec), "labels": T_(lab)}
    seqs = [np.concatenate([s, SEP_IDS, t, [EOS_ID]]) for s, t in zip(srcs, tgts)]
    L = max(len(x) for x in seqs) - 1
    idx = np.full((len(items), L), PAD_ID, np.int64); lab = np.full((len(items), L), -100, np.int64); plen = []
    for k, (x, s) in enumerate(zip(seqs, srcs)):
        n = len(x) - 1; idx[k, :n] = x[:-1]; P = len(s) + len(SEP_IDS)
        lab[k, P - 1:n] = x[P:]; plen.append(P)
    out = {"idx": T_(idx), "labels": T_(lab)}
    if arch == "prefixlm":
        i = torch.arange(L, device=device); P = torch.tensor(plen, device=device)
        causal = i[None, :] <= i[:, None]
        out["mask"] = (causal[None] | ((i[None, :, None] < P[:, None, None]) & (i[None, None, :] < P[:, None, None])))[:, None]
    return out

## 5. Building a model: Phase-1 weights → (smaller student) → adaptation method

In [7]:
_P1_CACHE, _P1_LOCK = {}, threading.Lock()
def load_phase1(key):
    with _P1_LOCK:
        if key not in _P1_CACHE:
            d = P1_DIRS[key]; assert d is not None, f"Phase-1 weights for {key} not found"
            info = load_json(d / "run_info.json") or load_json(d / "summary.json") or {}
            pcfg = dict(info.get("cfg") or {}); assert pcfg
            sd = torch.load(d / "weights_final.pt", map_location="cpu", weights_only=False)
            _P1_CACHE[key] = (pcfg, {k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
        return _P1_CACHE[key]

def apply_method_p6(model, cfg):
    m = cfg["method"]
    for p in model.parameters(): p.requires_grad = (m == "full")
    if m == "full": return
    if m in ("lora", "lorma"):
        apply_method(model, dict(cfg, method=m)); return
    if m in ("houlsby", "pfeiffer"):
        for parent, name, lin, is_out in target_linears(model, "all"):
            if not is_out: continue
            if m == "pfeiffer" and not isinstance(parent, MLP_TYPES): continue
            setattr(parent, name, AdapterLinear(lin, cfg["adapter_dim"]))
        return
    raise ValueError(m)

def build_model(cfg, device):
    pcfg, sd = load_phase1(cfg["model"])
    torch.manual_seed(cfg["seed"])
    lm = MODEL_REGISTRY[pcfg["arch"]](vocab=sd["wte.weight"].shape[0], **{k: v for k, v in pcfg.items() if k not in ("vocab", "dropout")}, dropout=cfg["dropout"])
    if cfg["init"] == "pretrained": lm.load_state_dict(sd)
    k = cfg.get("keep_layers")
    if k:
        if pcfg["arch"] == "encdec": lm.enc, lm.dec = nn.ModuleList(list(lm.enc)[:k // 2]), nn.ModuleList(list(lm.dec)[:k // 2])
        else:
            lm.blocks = nn.ModuleList(list(lm.blocks)[:k])
            if pcfg["arch"] == "efficient": lm.windows = lm.windows[:k]
    model = S2S(lm, pcfg["arch"])
    for mod in model.modules():
        if isinstance(mod, nn.Dropout): mod.p = cfg["dropout"]
        if isinstance(mod, (CausalSelfAttention, ModernAttention)): mod.dropout = cfg["dropout"]
    apply_method_p6(model, cfg)
    return model.to(device), pcfg

def count_parameters(model):
    seen, total, train = set(), 0, 0
    for p in model.parameters():
        if id(p) in seen: continue
        seen.add(id(p)); total += p.numel(); train += p.numel() if p.requires_grad else 0
    return total, train

def load_trained(name, device):
    cp, w = find_exp_file(name, "config.json"), find_exp_file(name, "trainable.pt")
    assert cp is not None and w is not None, f"p6/exps/{name}/(config.json, trainable.pt) not found — attach the run that trained {name}"
    tcfg = dict(DEFAULTS); tcfg.update(load_json(cp))
    m, _ = build_model(tcfg, device)
    sd = torch.load(w, map_location="cpu", weights_only=False)
    res = m.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()}, strict=False)
    assert not res.unexpected_keys, res.unexpected_keys[:5]
    return m, tcfg

## 6. Generation (KV cache) and document assembly
* Decoder / prefix-LM: the batch of prompts (`chunk Summary:`) is **left-padded**; one forward pass fills the KV cache
  (E: bidirectional over the prompt, exactly like its pretraining prefix), then one token per step.
* Encoder-decoder: encode the chunk once, decode from `<bos>` with a self-attention cache.
* Greedy decoding, **no repeated 4-grams**, repetition penalty 1.1, `<eos>` forbidden before `min_new` tokens.
* **Length control**: the length model gives the expected summary length W of a document (P6_01:
  `W ≈ a·doc_words^b`); each chunk gets a token budget proportional to its share of the document
  (`min_frac`·share … `max_frac`·share); the joined summary is cut at `len_scale`·W words.

In [8]:
@torch.no_grad()
def generate_ids(model, prompts, min_new, max_new, cfg, device):
    """prompts: list of src id arrays; min_new / max_new: per-prompt ints -> list of generated id lists"""
    model.eval(); lm = model.lm; out = [None] * len(prompts)
    amp = torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else torch.autocast("cpu", enabled=False)
    order = np.argsort([len(p) for p in prompts]); nr, rp = cfg["no_repeat_ngram"], cfg["rep_penalty"]
    for b0 in range(0, len(order), cfg["gen_batch"]):
        bi = order[b0:b0 + cfg["gen_batch"]]; B = len(bi)
        mx = int(max(max_new[i] for i in bi)); mn = torch.tensor([min_new[i] for i in bi], device=device)
        if model.enc_dec:
            rows = [prompts[i][:cfg["src_tok"]] for i in bi]; L = max(len(r) for r in rows)
            idx = torch.full((B, L), PAD_ID, dtype=torch.long); valid = torch.zeros((B, L), dtype=torch.bool)
            for r, x in enumerate(rows): idx[r, :len(x)] = torch.tensor(x); valid[r, :len(x)] = True
            idx, valid = idx.to(device), valid.to(device)
            with amp:
                mem = lm.encode(idx, valid)
                logits, cache = lm.decode(torch.full((B, 1), BOS_ID, dtype=torch.long, device=device), mem, valid, 0, None, use_cache=True)
        else:
            rows = [np.concatenate([prompts[i][:cfg["src_tok"]], SEP_IDS]) for i in bi]; L = max(len(r) for r in rows)
            idx = torch.full((B, L), PAD_ID, dtype=torch.long); valid = torch.zeros((B, L), dtype=torch.bool)
            for r, x in enumerate(rows): idx[r, L - len(x):] = torch.tensor(x); valid[r, L - len(x):] = True
            idx, valid = idx.to(device), valid.to(device)
            pos = (valid.long().cumsum(1) - 1).clamp(min=0); eye = torch.eye(L, dtype=torch.bool, device=device)[None]
            if model.arch == "prefixlm": mask = (valid[:, None, :] & valid[:, :, None]) | eye
            else: mask = (torch.tril(torch.ones(L, L, dtype=torch.bool, device=device))[None] & valid[:, None, :]) | eye
            with amp:
                if model.arch == "prefixlm": logits, _, cache = ModernLM.forward(lm, idx, None, pos, mask[:, None], None, True, True)
                else: logits, _, cache = lm(idx, pos=pos, mask=mask[:, None], use_cache=True, last_only=True)
            cur = pos[:, -1].clone()
        gen = [[] for _ in range(B)]; done = torch.zeros(B, dtype=torch.bool, device=device)
        seen = torch.zeros((B, logits.size(-1)), dtype=torch.bool, device=device); tables = [defaultdict(set) for _ in range(B)]
        for step in range(mx):
            lg = logits[:, -1].float()
            lg[:, 1:16] = -float("inf")                                       # pad / bos / reserved
            lg[:, EOS_ID] = torch.where(step < mn, torch.full_like(lg[:, 0], -float("inf")), lg[:, EOS_ID])
            if rp != 1.0: lg = torch.where(seen, torch.where(lg > 0, lg / rp, lg * rp), lg)
            if nr > 1 and step >= nr - 1:
                for r in range(B):
                    ban = tables[r].get(tuple(gen[r][-(nr - 1):]))
                    if ban: lg[r, list(ban)] = -float("inf")
            nxt = lg.argmax(-1)
            nl = nxt.tolist()
            for r in range(B):
                if done[r]: continue
                if nl[r] == EOS_ID or len(gen[r]) >= max_new[bi[r]]: done[r] = True; continue
                gen[r].append(nl[r]); seen[r, nl[r]] = True
                if nr > 1 and len(gen[r]) >= nr: tables[r][tuple(gen[r][-nr:-1])].add(gen[r][-1])
            if bool(done.all()): break
            with amp:
                if model.enc_dec:
                    logits, cache = lm.decode(nxt[:, None], mem, valid, step + 1, cache, use_cache=True)
                else:
                    valid = torch.cat([valid, torch.ones((B, 1), dtype=torch.bool, device=device)], 1); cur = cur + 1
                    if model.arch == "prefixlm": logits, _, cache = ModernLM.forward(lm, nxt[:, None], None, cur[:, None], valid[:, None, None, :], cache, True, False)
                    else: logits, _, cache = lm(nxt[:, None], pos=cur[:, None], mask=valid[:, None, None, :], cache=cache, use_cache=True)
        for r, i in enumerate(bi): out[i] = gen[r]
    return out

def expected_words(doc_words): return float(np.clip(LEN_MODEL["a"] * max(doc_words, 10) ** LEN_MODEL["b"], 30, 3000))

def summarize_split(model, pairs, cfg, device, doc_ids=None):
    """summaries for the documents of a split (all, or the given document indices) -> {doc_index: text}"""
    docs = np.arange(len(pairs.ids)) if doc_ids is None else np.asarray(doc_ids)
    wanted = set(docs.tolist()); items = [k for k in range(len(pairs)) if pairs.doc[k] in wanted]
    doc_tok = defaultdict(int)
    for k in items: doc_tok[pairs.doc[k]] += pairs.src_len(k)
    prompts, mins, maxs = [], [], []
    for k in items:
        d = pairs.doc[k]; share = pairs.src_len(k) / max(1, doc_tok[d])
        budget = expected_words(pairs.doc_words[d]) * 1.35 * share            # ≈ 1.35 tokens per word
        mins.append(int(max(4, cfg["min_frac"] * budget))); maxs.append(int(np.clip(cfg["max_frac"] * budget, 24, cfg["tgt_tok"])))
        prompts.append(pairs.s(k))
    gen = generate_ids(model, prompts, mins, maxs, cfg, device)
    parts = defaultdict(list)
    for k, g in zip(items, gen): parts[pairs.doc[k]].append((pairs.chunk[k], TOK.decode(g).strip()))
    out = {}
    for d in docs:
        text = " ".join(t for _, t in sorted(parts[d]) if t)
        words = text.split(); cap = int(cfg["len_scale"] * expected_words(pairs.doc_words[d]))
        out[int(d)] = " ".join(words[:cap]) if words else "The appeal is disposed of."
    return out

## 7. Metrics, training, experiment runner
ROUGE as in the Phase-1 evaluator (whitespace tokens): ROUGE-1 F1 and ROUGE-L F1 (LCS). **Redundancy** = share of
repeated word 3-grams in a summary. BERTScore (primary metric) is computed for every experiment at the end (cell 10).
**KD** (Hinton et al. 2015, word-level as in Kim & Rush 2016): the teacher reads the same `chunk Summary: summary`
sequence; on every summary token the student matches the teacher's temperature-softened next-token distribution:
`α·CE(gold) + (1−α)·T²·KL(p_teacher,T ‖ p_student,T)` — possible because all Phase-1 models share one tokenizer.

In [9]:
def lcs_length(a, b):
    if not a or not b: return 0
    if len(a) < len(b): a, b = b, a
    masks = {}
    for j, t in enumerate(a): masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0); x = m | state; y = (state << 1) | 1; state = x & ~(x - y)
    return bin(state).count("1")
def rouge_l_f1(p, r):
    pt, rt = p.split(), r.split()
    if not pt or not rt: return 0.0
    l = lcs_length(pt, rt)
    return 0.0 if not l else 2 * (l / len(pt)) * (l / len(rt)) / (l / len(pt) + l / len(rt))
def rouge1_f1(p, r):
    pc, rc = Counter(p.split()), Counter(r.split()); ov = sum((pc & rc).values())
    if not ov: return 0.0
    pp, rr = ov / sum(pc.values()), ov / sum(rc.values()); return 2 * pp * rr / (pp + rr)
def redundancy(t):
    w = t.lower().split(); g = [tuple(w[i:i + 3]) for i in range(len(w) - 2)]
    return 0.0 if not g else 1 - len(set(g)) / len(g)
def val_scores(preds):
    refs = [VAL_REFS[i] for i in preds]; ps = list(preds.values())
    return {"ROUGE-1": float(np.mean([rouge1_f1(p, r) for p, r in zip(ps, refs)])), "ROUGE-L": float(np.mean([rouge_l_f1(p, r) for p, r in zip(ps, refs)])),
            "words": float(np.mean([len(p.split()) for p in ps])), "ref_words": float(np.mean([len(r.split()) for r in refs])),
            "redundancy_3gram": float(np.mean([redundancy(p) for p in ps])), "ref_redundancy_3gram": float(np.mean([redundancy(r) for r in refs]))}

def train_items(pairs, cfg):
    rng = np.random.default_rng(cfg["seed"]); ks = np.arange(len(pairs))
    if cfg["drop_empty_targets"] and pairs.tgt is not None: ks = ks[np.diff(pairs.tgt_off)[ks] > 0]
    if cfg["max_train_pairs"] and len(ks) > cfg["max_train_pairs"]: ks = np.sort(rng.choice(ks, cfg["max_train_pairs"], replace=False))
    return ks

@torch.no_grad()
def val_loss(model, cfg, device, n=600):
    model.eval(); p = PAIRS["val"]; ks = train_items(p, dict(cfg, max_train_pairs=n)); tot, cnt = 0.0, 0
    amp = torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else torch.autocast("cpu", enabled=False)
    for i in range(0, len(ks), cfg["batch"]):
        b = make_batch(p, [(p.s(k), p.t(k)) for k in ks[i:i + cfg["batch"]]], cfg, model.arch, device)
        with amp: lg = model.logits(b)
        m = b["labels"] != -100
        tot += F.cross_entropy(lg.float()[m], b["labels"][m], reduction="sum").item(); cnt += int(m.sum())
    return tot / max(cnt, 1)

def train_model(cfg, device, log):
    p = PAIRS["train"]; ks = train_items(p, cfg)
    teacher = None
    if cfg["teacher"]:
        teacher, tcfg = load_trained(cfg["teacher"], device); teacher.eval()
        for q in teacher.parameters(): q.requires_grad = False
        log(f"teacher {cfg['teacher']} ({count_parameters(teacher)[0]/1e6:.1f}M params, arch {teacher.arch})")
    model, pcfg = build_model(cfg, device); total, trainable = count_parameters(model)
    params = [q for q in model.parameters() if q.requires_grad]
    opt = torch.optim.AdamW([{"params": [q for q in params if q.ndim >= 2], "weight_decay": cfg["weight_decay"]},
                             {"params": [q for q in params if q.ndim < 2], "weight_decay": 0.0}], lr=cfg["lr"], betas=(0.9, 0.98))
    total_upd = max(1, math.ceil(len(ks) / (cfg["batch"] * cfg["accum"])) * cfg["epochs"]); warm = max(1, int(cfg["warmup"] * total_upd))
    use_amp = device.type == "cuda"; scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    amp = torch.autocast("cuda", dtype=torch.float16) if use_amp else torch.autocast("cpu", enabled=False)
    log(f"model {cfg['model']} ({pcfg['arch']}{', ' + str(cfg['keep_layers']) + ' layers' if cfg['keep_layers'] else ''}) method={cfg['method']} | "
        f"params {total/1e6:.1f}M, trainable {trainable/1e6:.2f}M ({100*trainable/total:.2f}%) | {len(ks)} training pairs, {total_upd} updates")
    if use_amp: torch.cuda.reset_peak_memory_stats(device)
    rng = np.random.default_rng(cfg["seed"]); upd, t0, hist, best = 0, time.time(), [], (1e9, None, 0)
    for ep in range(1, cfg["epochs"] + 1):
        model.train(); order = rng.permutation(ks); run, nb = 0.0, 0; opt.zero_grad(set_to_none=True)
        for i in range(0, len(order), cfg["batch"]):
            b = make_batch(p, [(p.s(k), p.t(k)) for k in order[i:i + cfg["batch"]]], cfg, model.arch, device)
            m = b["labels"] != -100
            with amp: lg = model.logits(b)
            s_lg = lg.float()[m]; loss = F.cross_entropy(s_lg, b["labels"][m])
            if teacher is not None:
                # same token sequences, same order of summary tokens -> teacher and student label positions line up
                tb = b if teacher.arch == model.arch else make_batch(p, [(p.s(k), p.t(k)) for k in order[i:i + cfg["batch"]]], cfg, teacher.arch, device)
                with torch.no_grad(), amp: t_lg = teacher.logits(tb).float()[tb["labels"] != -100]
                assert t_lg.shape == s_lg.shape, (t_lg.shape, s_lg.shape)
                T, a = cfg["kd_T"], cfg["kd_alpha"]
                kd = F.kl_div(F.log_softmax(s_lg / T, -1), F.log_softmax(t_lg / T, -1), log_target=True, reduction="batchmean") * T * T
                loss = a * loss + (1 - a) * kd
            scaler.scale(loss / cfg["accum"]).backward(); run += loss.item(); nb += 1
            if nb % cfg["accum"] == 0 or i + cfg["batch"] >= len(order):
                lr_mult = (upd + 1) / warm if upd < warm else max(0.0, (total_upd - upd) / max(1, total_upd - warm))
                for g in opt.param_groups: g["lr"] = cfg["lr"] * lr_mult
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, cfg["grad_clip"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); upd += 1
            if nb % 500 == 0: log(f"  epoch {ep} step {nb}: loss {run / nb:.4f} ({(time.time() - t0)/60:.1f} min)")
        vl = val_loss(model, cfg, device)
        msg = {"epoch": ep, "train_loss": run / max(nb, 1), "val_loss": vl, "min": (time.time() - t0) / 60}
        hist.append(msg); log(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in msg.items()}))
        if vl < best[0]:
            best = (vl, {n: q.detach().to("cpu", copy=True) for n, q in model.named_parameters() if q.requires_grad}, ep)
    model.load_state_dict(best[1], strict=False)
    del teacher
    info = {"total_params": total, "trainable_params": trainable, "train_min": (time.time() - t0) / 60, "history": hist, "best_epoch": best[2],
            "best_val_loss": best[0], "peak_mem_gb": torch.cuda.max_memory_allocated(device) / 1e9 if use_amp else None, "arch": pcfg["arch"]}
    return model, info, set(best[1])

def save_trainable(model, names, path):
    sd = {n: q.detach().half().cpu() if q.is_floating_point() else q.detach().cpu() for n, q in model.named_parameters() if n in names}
    torch.save(sd, str(path) + ".tmp"); os.replace(str(path) + ".tmp", path)

def write_submission(preds, d):
    test = PAIRS["test"]; obj = [{"id": test.ids[i], "summary": preds[i]} for i in range(len(test.ids))]
    assert len(obj) == len(test.ids) and all(o["summary"].strip() for o in obj)
    save_json(obj, d / "test_predictions.json")
    with zipfile.ZipFile(d / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z: z.writestr("prediction.json", json.dumps(obj, ensure_ascii=False))

def run_experiment(name, device):
    cfg = exp_cfg(name); d = EXPS_DIR / name; d.mkdir(parents=True, exist_ok=True)
    if (d / "result.json").exists(): print(f"[{name}] already done — skipped"); return
    logf = open(d / "log.txt", "a")
    def log(msg):
        line = f"[{name} | {device}] {msg}"; print(line, flush=True); logf.write(line + "\n"); logf.flush()
    save_json(cfg, d / "config.json")
    if cfg.get("gen_from"):                                      # decoding-only variant of a finished experiment (no training)
        model, scfg = load_trained(cfg["gen_from"], device)
        src_res = load_json(find_exp_file(cfg["gen_from"], "result.json"))
        info = dict(src_res["info"], train_min=0.0); cfg = dict(scfg, **{k: v for k, v in EXPERIMENTS[name].items() if k != "gen_from"}, name=name, gen_from=cfg["gen_from"])
        log(f"decoding-only variant of {cfg['gen_from']}: " + json.dumps({k: cfg[k] for k in ("len_scale", "no_repeat_ngram", "rep_penalty", "min_frac", "max_frac")}))
    else:
        model, info, names = train_model(cfg, device, log)
        save_trainable(model, names, d / "trainable.pt")
    rng = np.random.default_rng(1); nv = min(cfg["n_val_gen"], len(PAIRS["val"].ids))
    vdocs = np.sort(rng.choice(len(PAIRS["val"].ids), nv, replace=False))
    t0 = time.time(); vp = summarize_split(model, PAIRS["val"], cfg, device, vdocs); gen_s = (time.time() - t0) / nv
    vpred = {PAIRS["val"].ids[k]: v for k, v in vp.items()}; save_json(vpred, d / "val_predictions.json")
    sc = val_scores(vpred); log(f"validation ({nv} docs, {gen_s:.2f} s/doc): " + json.dumps({k: round(v, 4) for k, v in sc.items()}))
    t0 = time.time(); tp = summarize_split(model, PAIRS["test"], cfg, device); test_min = (time.time() - t0) / 60
    write_submission(tp, d)
    row = {"name": name, "model": cfg["model"] + (f"[{cfg['keep_layers']}L]" if cfg.get("keep_layers") else ""), "arch": info["arch"],
           "method": cfg["method"] + (f"+KD({cfg['teacher']})" if cfg["teacher"] else "") + (f" [decode: len×{cfg['len_scale']}, no-repeat {cfg['no_repeat_ngram']}]" if cfg.get("gen_from") else ""), **sc, "val_loss": info["best_val_loss"],
           "total_params_M": info["total_params"] / 1e6, "trainable_params_M": info["trainable_params"] / 1e6,
           "trainable_%": 100 * info["trainable_params"] / info["total_params"], "peak_mem_GB": info["peak_mem_gb"], "train_min": info["train_min"],
           "gen_sec_per_doc": gen_s, "test_gen_min": test_min, "best_epoch": info["best_epoch"]}
    save_json({"row": row, "info": info, "cfg": cfg}, d / "result.json")
    log(f"DONE ROUGE-1 {sc['ROUGE-1']:.4f} ROUGE-L {sc['ROUGE-L']:.4f} | {sc['words']:.0f} words (ref {sc['ref_words']:.0f}) | test summaries in {test_min:.1f} min")
    logf.close(); del model
    if device.type == "cuda": torch.cuda.empty_cache()

## 8. Unit tests (CPU, seconds)

In [10]:
def _tests():
    base = nn.Linear(8, 12, bias=False); x = torch.randn(3, 8)
    for mod in (LoRMALinear(base, 4, 8), LoRALinear(base, 4, 8), AdapterLinear(base, 6)): assert torch.allclose(mod(x), base(x), atol=1e-5)
    assert abs(rouge_l_f1("a b c d", "a b c d") - 1) < 1e-9 and abs(rouge1_f1("a b", "b c") - 0.5) < 1e-9 and redundancy("a b c a b c") > 0
    s, t = np.array([5, 6, 7]), np.array([8, 9])
    b = make_batch(None, [(s, t)], DEFAULTS, "gpt2", torch.device("cpu"))
    lab = b["labels"][0].tolist(); idx = b["idx"][0].tolist()
    assert [v for v in lab if v != -100] == [8, 9, EOS_ID] and idx[:3] == [5, 6, 7], (idx, lab)
    pos_first = lab.index(8); assert idx[pos_first] == SEP_IDS[-1], "the summary's first token is predicted from the separator"
    b = make_batch(None, [(s, t), (s[:1], t[:1])], DEFAULTS, "prefixlm", torch.device("cpu"))
    m = b["mask"][0, 0]; P = 3 + len(SEP_IDS)
    assert m[0, P - 1] and not m[0, P] and m[P, 0], "prefix is bidirectional, summary causal"
    b = make_batch(None, [(s, t)], DEFAULTS, "encdec", torch.device("cpu"))
    assert b["dec_in"][0].tolist()[:3] == [BOS_ID, 8, 9] and b["labels"][0].tolist()[:3] == [8, 9, EOS_ID]
    print("unit tests passed")
_tests()

unit tests passed


## 9. Run the queue (one worker thread per GPU; a KD student waits for its teacher if both are in the queue)

In [11]:
def run_queue(names):
    def dep_of(n): c = exp_cfg(n); return c.get("gen_from") or c["teacher"]
    missing = [f"{n} needs {dep_of(n)}" for n in names if dep_of(n) and dep_of(n) not in names
               and not (EXPS_DIR / n / "result.json").exists() and find_exp_file(dep_of(n), "trainable.pt") is None]
    if missing: print("WARNING — these will fail, attach the runs that trained:\n  " + "\n  ".join(missing))
    pending, running, failed, lock = list(names), set(), set(), threading.Lock()
    def take():
        with lock:
            for n in pending:
                dep = dep_of(n)
                if dep in failed: pending.remove(n); failed.add(n); print(f"[{n}] not run: {dep} failed"); return "retry"
                if dep and (dep in pending or dep in running): continue
                pending.remove(n); running.add(n); return n
            return "wait" if pending else None
    devices = [torch.device(f"cuda:{i}") for i in range(NUM_GPUS)] or [torch.device("cpu")]
    def worker(dev):
        while True:
            name = take()
            if name is None: return
            if name in ("wait", "retry"):
                if name == "wait": time.sleep(20)
                continue
            ok = False
            if not (EXPS_DIR / name / "result.json").exists() and time_left_h() < exp_cfg(name)["est_h"] + 0.2:
                print(f"[{name}] skipped: not enough session time left")
            else:
                try:
                    if dev.type == "cuda":
                        with torch.cuda.device(dev): run_experiment(name, dev)
                    else: run_experiment(name, dev)
                    ok = (EXPS_DIR / name / "result.json").exists()
                except Exception:
                    print(f"[{name}] FAILED:\n{traceback.format_exc()}", flush=True)
                    (EXPS_DIR / name).mkdir(parents=True, exist_ok=True); (EXPS_DIR / name / "error.txt").write_text(traceback.format_exc())
            with lock:
                running.discard(name)
                if not ok: failed.add(name)
    threads = [threading.Thread(target=worker, args=(dv,)) for dv in devices]
    for t in threads: t.start()
    for t in threads: t.join()

t_q = time.time(); run_queue(RUN_QUEUE); print(f"queue finished in {(time.time() - t_q)/60:.1f} min")

[E_full | cuda:1] model E (prefixlm) method=full | params 110.1M, trainable 110.12M (100.00%) | 20000 training pairs, 1250 updates[F_houlsby | cuda:0] model F (efficient) method=houlsby | params 110.1M, trainable 2.38M (2.16%) | 20000 training pairs, 1250 updates

[F_houlsby | cuda:0]   epoch 1 step 500: loss 1.7778 (1.6 min)
[E_full | cuda:1]   epoch 1 step 500: loss 1.9065 (2.2 min)
[F_houlsby | cuda:0]   epoch 1 step 1000: loss 1.6286 (3.3 min)
[E_full | cuda:1]   epoch 1 step 1000: loss 1.7596 (4.5 min)
[F_houlsby | cuda:0]   epoch 1 step 1500: loss 1.5635 (5.0 min)
[F_houlsby | cuda:0]   epoch 1 step 2000: loss 1.5291 (6.6 min)
[E_full | cuda:1]   epoch 1 step 1500: loss 1.6934 (6.7 min)
[F_houlsby | cuda:0]   epoch 1 step 2500: loss 1.4943 (8.3 min)
[E_full | cuda:1]   epoch 1 step 2000: loss 1.6581 (9.0 min)
[F_houlsby | cuda:0]   epoch 1 step 3000: loss 1.4725 (9.9 min)
[E_full | cuda:1]   epoch 1 step 2500: loss 1.6219 (11.3 min)
[F_houlsby | cuda:0]   epoch 1 step 3500: loss 

## 10. Results + BERTScore on validation
BERTScore F1 exactly as the Phase-1 evaluator computes it (`bert_score.score(..., lang="en")`, RoBERTa-large). It also
scores the **lead-N baseline** (first N words of the document) for reference. Needs Internet (model download).

In [12]:
rows = [load_json(f)["row"] for f in sorted(EXPS_DIR.glob("*/result.json"))]
BS = load_json(ROOT / "p6" / "bertscore_val.json", {})
if BERTSCORE_EVAL and rows:
    try:
        import importlib, subprocess
        if importlib.util.find_spec("bert_score") is None: subprocess.run([sys.executable, "-m", "pip", "-q", "install", "bert-score"], check=True)
        import bert_score
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        vtext = load_json(find_artifact("p6/data/val_docs_text.json")); vdoc = dict(zip(vtext["ids"], vtext["docs"]))
        for f in sorted(EXPS_DIR.glob("*/val_predictions.json")):
            name = f.parent.name
            if name in BS: continue
            pr = load_json(f); ids = list(pr)
            _, _, f1 = bert_score.score([pr[i] for i in ids], [VAL_REFS[i] for i in ids], lang="en", batch_size=8, verbose=False, device=dev)
            BS[name] = float(f1.mean())
            if "lead-N" not in BS:
                lead = [" ".join(vdoc[i].split()[:int(expected_words(len(vdoc[i].split())))]) for i in ids]
                _, _, f1 = bert_score.score(lead, [VAL_REFS[i] for i in ids], lang="en", batch_size=8, verbose=False, device=dev)
                BS["lead-N"] = float(f1.mean())
        save_json(BS, ROOT / "p6" / "bertscore_val.json")
    except Exception as e:
        print("BERTScore skipped:", repr(e)[:300])
if rows:
    df = pd.DataFrame(rows); df["BERTScore"] = df["name"].map(BS)
    df = df.sort_values(["BERTScore", "ROUGE-L"], ascending=False)
    df.to_csv(ROOT / "p6" / "p6_experiments.csv", index=False)
    cols = ["name", "model", "method", "BERTScore", "ROUGE-1", "ROUGE-L", "words", "ref_words", "redundancy_3gram", "val_loss",
            "trainable_%", "total_params_M", "peak_mem_GB", "train_min", "gen_sec_per_doc", "best_epoch"]
    display(df[[c for c in cols if c in df]].round(4).to_string(index=False))
    if "lead-N" in BS: print(f"lead-N baseline BERTScore on the same validation documents: {BS['lead-N']:.4f}")
    best = df.iloc[0]["name"]; print(f"\nbest: {best} -> {ROOT}/p6/exps/{best}/submission.zip")
    ex = load_json(EXPS_DIR / best / "val_predictions.json"); k = next(iter(ex))
    print("\nexample (validation):\nGENERATED:", " ".join(ex[k].split()[:150]), "\nREFERENCE:", " ".join(VAL_REFS[k].split()[:150]))
for f in sorted(EXPS_DIR.glob("*/error.txt")): print(f"FAILED: {f.parent.name}\n{f.read_text()[-1500:]}")

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (9 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


'         name  model   method  BERTScore  ROUGE-1  ROUGE-L    words  ref_words  redundancy_3gram  val_loss  trainable_%  total_params_M  peak_mem_GB  train_min  gen_sec_per_doc  best_epoch\n       E_full      E     full     0.8545   0.5216   0.2717 653.5133     746.28            0.0821    1.2571     100.0000        110.1197       5.4813    45.6222           1.7483           2\n    F_houlsby      F  houlsby     0.8532   0.4975   0.2597 571.7933     746.28            0.0799    1.2305       2.1602        110.1396       3.6186    33.5029           1.4824           2\nA0_6L_control A0[6L]     full     0.8507   0.5080   0.2555 673.3267     746.28            0.0865    1.5854     100.0000         68.4810       3.1968    18.3065           0.8695           2\n   B_pfeiffer      B pfeiffer     0.8392   0.4227   0.2074 511.0133     746.28            0.1877    2.4613       0.9481        125.4705       2.3792    19.1558           0.9175           2'

lead-N baseline BERTScore on the same validation documents: 0.8337

best: E_full -> /kaggle/working/cs781_p1/p6/exps/E_full/submission.zip

example (validation):
GENERATED: The assessee despatched some part of the manufactured goods outside the State for sale on consignment basis. However, the assessee had not paid the taxes on the taxable raw material consumed in the manufacture of such goods. The assessee had not pay the taxes on such raw material consumed. The assessee disputed its liability to pay penalty and interest on the amount of tax withheld on the plea that there was no wilful or intentional default on the part of the assessee to pay the taxes due under section 4B of the Act as the assesee was under a bona fide belief that no tax was to be paid on the raw material purchased for the manufacture of paper which was ultimately sent outside the State on consignment basis, and hence the assessee could not take shelter under the said two decisions. Des Raj Pushpak Kumar 's case 58 